# Polynomial Regression and Regularization: Modeling Air Quality and Health Outcomes

In this tutorial, we explore how **polynomial regression** and **regularization techniques** (LASSO and Ridge) can help us model complex, non-linear relationships in environmental data.

Our scenario: we have measurements of fine particulate matter concentration (PM2.5) from air quality sensors across a metropolitan area. We want to predict a **respiratory health index** — a composite score reflecting hospital admissions, inhaler prescriptions, and clinic visits — from pollution levels. The relationship between pollution and health impact is unlikely to be a simple straight line: at low concentrations the effect may be modest, but as PM2.5 climbs, health outcomes can deteriorate rapidly.

We will:
1. Simulate realistic air quality data with a known non-linear pattern
2. Show that ordinary linear regression falls short
3. Fit polynomial models of increasing complexity and watch overfitting emerge
4. Use LASSO (L1) and Ridge (L2) regularization to tame that complexity
5. Select the best model using cross-validation

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

print('All imports successful.')

---
## 2. Simulating Air Quality Data

We generate 200 observations. The feature `pm25_concentration` ranges from about 5 to 80 µg/m³ (a realistic span from clean air to unhealthy levels). The target `respiratory_index` follows a non-linear function of PM2.5 — specifically a combination of a quadratic trend and a mild logarithmic term — plus Gaussian noise representing natural variability in health outcomes.

In [ ]:
n_samples = 200

# PM2.5 concentration in µg/m³, drawn from a skewed distribution
pm25 = np.random.exponential(scale=20, size=n_samples) + 5
pm25 = np.clip(pm25, 5, 80)  # keep within realistic bounds

# True underlying relationship (unknown to the "modeler")
# A quadratic-like curve: health impact accelerates at higher pollution
def true_relationship(x):
    return 10 + 0.8 * x + 0.012 * x**2 - 0.00005 * x**3 + 3 * np.log(x + 1)

respiratory_index = true_relationship(pm25) + np.random.normal(0, 5, size=n_samples)

# Assemble into a DataFrame
df = pd.DataFrame({
    'pm25_concentration': pm25,
    'respiratory_index': respiratory_index
})

print(f'Dataset shape: {df.shape}')
df.describe().round(2)

---
## 3. Visualizing the Data

Before fitting any models, let's look at what we're working with. A scatter plot should reveal that the relationship between PM2.5 and the respiratory index is **not** a straight line.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['pm25_concentration'], df['respiratory_index'],
           alpha=0.5, edgecolors='k', linewidths=0.5, color='steelblue', s=40)

# Overlay the true curve for reference
x_smooth = np.linspace(5, 80, 300)
ax.plot(x_smooth, true_relationship(x_smooth), 'r--', linewidth=2, label='True relationship')

ax.set_xlabel('PM2.5 Concentration (µg/m³)')
ax.set_ylabel('Respiratory Index')
ax.set_title('Air Quality vs. Respiratory Health')
ax.legend()
plt.tight_layout()
plt.show()

The curvature is visible — the index rises steeply at first and gradually bends. A straight line would miss this pattern entirely.

---
## 4. Train/Test Split

We hold out 30% of our data for testing, so every model is evaluated on points it has never seen.

In [ ]:
X = df[['pm25_concentration']].values
y = df['respiratory_index'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

---
## 5. Linear Regression Baseline

Let's start with the simplest model — ordinary least squares with no feature engineering — and see where it falls short.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train_lr = lr.predict(X_train)
y_pred_test_lr = lr.predict(X_test)

print('--- Linear Regression ---')
print(f'Train MSE: {mean_squared_error(y_train, y_pred_train_lr):.2f}')
print(f'Test MSE:  {mean_squared_error(y_test, y_pred_test_lr):.2f}')
print(f'Test R²:   {r2_score(y_test, y_pred_test_lr):.4f}')

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_test, y_test, alpha=0.5, edgecolors='k', linewidths=0.5,
           color='steelblue', s=40, label='Test data')
x_line = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
ax.plot(x_line, lr.predict(x_line), color='orange', linewidth=2, label='Linear fit')
ax.set_xlabel('PM2.5 Concentration (µg/m³)')
ax.set_ylabel('Respiratory Index')
ax.set_title('Linear Regression — A Poor Fit for Curved Data')
ax.legend()
plt.tight_layout()
plt.show()

The straight line systematically under-predicts at low and high PM2.5 values and over-predicts in the middle. We need more flexibility.

---
## 6. Polynomial Feature Engineering

The idea behind polynomial regression is straightforward: instead of fitting $y = \beta_0 + \beta_1 x$, we create new features $x^2, x^3, \ldots, x^d$ and fit:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$$

This is still a **linear** model in the coefficients — we're just giving it richer inputs. Scikit-learn's `PolynomialFeatures` handles the transformation for us.

In [ ]:
# Quick demonstration: what PolynomialFeatures actually produces
demo = np.array([[3], [7]])
pf = PolynomialFeatures(degree=3, include_bias=False)
transformed = pf.fit_transform(demo)

print('Original:')
print(demo)
print('\nAfter degree-3 polynomial expansion:')
print(transformed)
print(f'Feature names: {pf.get_feature_names_out()}')

---
## 7. Fitting Polynomial Models of Varying Degrees

We'll fit polynomials of degree 1, 2, 3, 5, and 10, then plot each one on top of the data. Watch how the curve becomes increasingly wiggly as the degree rises.

In [ ]:
degrees = [1, 2, 3, 5, 10]
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#e67e22']

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
axes = axes.flatten()

poly_results = {}

for idx, deg in enumerate(degrees):
    ax = axes[idx]

    # Build a pipeline: polynomial features -> linear regression
    model = make_pipeline(PolynomialFeatures(deg, include_bias=False),
                          LinearRegression())
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    poly_results[deg] = {'train_mse': train_mse, 'test_mse': test_mse, 'model': model}

    # Scatter + curve
    ax.scatter(X_train, y_train, alpha=0.3, s=20, color='gray', label='Train')
    ax.scatter(X_test, y_test, alpha=0.5, s=30, color='steelblue', edgecolors='k',
               linewidths=0.5, label='Test')

    x_plot = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
    ax.plot(x_plot, model.predict(x_plot), color=colors[idx], linewidth=2)

    ax.set_title(f'Degree {deg}\nTrain MSE={train_mse:.1f}  Test MSE={test_mse:.1f}')
    ax.set_xlabel('PM2.5')
    if idx % 3 == 0:
        ax.set_ylabel('Respiratory Index')

# Remove the unused 6th subplot
axes[-1].axis('off')

fig.suptitle('Polynomial Fits of Increasing Degree', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

Notice that the degree-2 and degree-3 fits track the curve well. By degree 10, the model starts to chase individual noise points — classic overfitting.

---
## 8. The Overfitting Curve: Train vs. Test Error by Degree

The hallmark of overfitting is a growing gap between training error and test error. Let's trace both as we increase polynomial degree from 1 through 15.

In [ ]:
max_degree = 15
train_errors = []
test_errors = []

for d in range(1, max_degree + 1):
    model = make_pipeline(PolynomialFeatures(d, include_bias=False),
                          LinearRegression())
    model.fit(X_train, y_train)
    train_errors.append(mean_squared_error(y_train, model.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, model.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, max_degree + 1), train_errors, 'o-', color='#2ecc71',
        linewidth=2, markersize=6, label='Train MSE')
ax.plot(range(1, max_degree + 1), test_errors, 's-', color='#e74c3c',
        linewidth=2, markersize=6, label='Test MSE')

ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('Mean Squared Error')
ax.set_title('Overfitting Curve: Train vs. Test Error')
ax.legend()
ax.set_yscale('log')
ax.set_xticks(range(1, max_degree + 1))
plt.tight_layout()
plt.show()

best_deg = np.argmin(test_errors) + 1
print(f'Lowest test MSE at degree {best_deg} (MSE = {min(test_errors):.2f})')

Training error keeps dropping (the model memorizes), but test error eventually climbs. The sweet spot is a low-degree polynomial — but what if we want to use a high-degree expansion *without* overfitting? That's where regularization comes in.

---
## 9. LASSO Regression — Taming Complexity with L1 Regularization

LASSO (Least Absolute Shrinkage and Selection Operator) adds a penalty proportional to the **absolute value** of the coefficients:

$$\min_\beta \frac{1}{2n} \|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

The key property of LASSO is **sparsity**: it drives some coefficients exactly to zero, effectively performing automatic feature selection. This is especially valuable when we've created many polynomial features but suspect only a few are truly needed.

Let's apply LASSO to a degree-10 polynomial expansion — the same one that overfit badly when unregularized.

In [ ]:
# Create degree-10 polynomial features
poly10 = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly10.fit_transform(X_train)
X_test_poly = poly10.transform(X_test)

# Scale features — essential for regularized models because the penalty
# treats all coefficients equally, so features must be on the same scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

print(f'Polynomial feature matrix shape: {X_train_scaled.shape}')
print(f'Features: {poly10.get_feature_names_out()}')

In [ ]:
# Fit LASSO with a moderate alpha
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

y_pred_train_lasso = lasso.predict(X_train_scaled)
y_pred_test_lasso = lasso.predict(X_test_scaled)

print('--- LASSO (alpha=1.0) on Degree-10 Features ---')
print(f'Train MSE: {mean_squared_error(y_train, y_pred_train_lasso):.2f}')
print(f'Test MSE:  {mean_squared_error(y_test, y_pred_test_lasso):.2f}')
print(f'Test R²:   {r2_score(y_test, y_pred_test_lasso):.4f}')
print(f'\nNon-zero coefficients: {np.sum(lasso.coef_ != 0)} out of {len(lasso.coef_)}')

# Show which coefficients survived
coef_df = pd.DataFrame({
    'Feature': poly10.get_feature_names_out(),
    'Coefficient': lasso.coef_
})
print('\nCoefficient values:')
print(coef_df.to_string(index=False))

LASSO has zeroed out many of the higher-order terms — it decided they were adding noise, not signal. The test MSE should be much better than the unregularized degree-10 fit.

---
## 10. LassoCV — Finding the Optimal Alpha Automatically

Choosing the right value of `alpha` (the regularization strength) is critical. Too large and we underfit; too small and we overfit. `LassoCV` performs k-fold cross-validation over a grid of alpha values and picks the one with the lowest CV error.

In [ ]:
alphas = np.logspace(-4, 2, 100)

lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=50000, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print(f'Optimal alpha: {lasso_cv.alpha_:.6f}')
print(f'Number of non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)}')

y_pred_test_cv = lasso_cv.predict(X_test_scaled)
print(f'\nTest MSE:  {mean_squared_error(y_test, y_pred_test_cv):.2f}')
print(f'Test R²:   {r2_score(y_test, y_pred_test_cv):.4f}')

In [ ]:
# Plot the cross-validation curve
# LassoCV stores the mean CV MSE for each alpha
mean_mse = np.mean(lasso_cv.mse_path_, axis=1)
std_mse = np.std(lasso_cv.mse_path_, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(lasso_cv.alphas_, mean_mse, 'o-', color='steelblue', markersize=3)
ax.fill_between(lasso_cv.alphas_, mean_mse - std_mse, mean_mse + std_mse,
                alpha=0.2, color='steelblue')
ax.axvline(lasso_cv.alpha_, color='red', linestyle='--', linewidth=2,
           label=f'Optimal alpha = {lasso_cv.alpha_:.4f}')

ax.set_xlabel('Alpha (regularization strength)')
ax.set_ylabel('Mean CV MSE')
ax.set_title('LassoCV: Cross-Validation Error vs. Alpha')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## 11. Coefficient Paths — How LASSO Shrinks Features

A powerful way to visualize LASSO's behavior is the **coefficient path**: we plot each coefficient as a function of alpha. As alpha increases, coefficients shrink toward zero — and the least important ones vanish first.

In [ ]:
# Fit LASSO for many alpha values and record coefficients
coef_paths = []
path_alphas = np.logspace(-4, 2, 80)

for a in path_alphas:
    lasso_temp = Lasso(alpha=a, max_iter=50000)
    lasso_temp.fit(X_train_scaled, y_train)
    coef_paths.append(lasso_temp.coef_.copy())

coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(10, 6))
feature_names = poly10.get_feature_names_out()

for i in range(coef_paths.shape[1]):
    ax.semilogx(path_alphas, coef_paths[:, i], linewidth=1.5, label=feature_names[i])

ax.axvline(lasso_cv.alpha_, color='black', linestyle='--', linewidth=1,
           label=f'Optimal alpha = {lasso_cv.alpha_:.4f}')
ax.set_xlabel('Alpha')
ax.set_ylabel('Coefficient value')
ax.set_title('LASSO Coefficient Paths')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

The low-order terms (x, x²) tend to be the last ones standing as alpha increases, confirming that they carry the most predictive signal.

---
## 12. Ridge vs. LASSO Comparison

**Ridge regression** uses an L2 penalty ($\alpha \|\beta\|_2^2$) instead of L1. The crucial difference: Ridge shrinks coefficients toward zero but rarely makes them *exactly* zero. It keeps all features in the model.

Let's compare both approaches on our degree-10 polynomial features.

In [ ]:
# Ridge with cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 100), cv=5)
ridge_cv.fit(X_train_scaled, y_train)

y_pred_test_ridge = ridge_cv.predict(X_test_scaled)

print(f'Ridge optimal alpha: {ridge_cv.alpha_:.4f}')
print(f'Ridge Test MSE:  {mean_squared_error(y_test, y_pred_test_ridge):.2f}')
print(f'Ridge Test R²:   {r2_score(y_test, y_pred_test_ridge):.4f}')
print(f'Ridge non-zero coefficients: {np.sum(ridge_cv.coef_ != 0)} / {len(ridge_cv.coef_)}')
print()
print(f'LASSO optimal alpha: {lasso_cv.alpha_:.4f}')
print(f'LASSO Test MSE: {mean_squared_error(y_test, y_pred_test_cv):.2f}')
print(f'LASSO Test R²:  {r2_score(y_test, y_pred_test_cv):.4f}')
print(f'LASSO non-zero coefficients: {np.sum(lasso_cv.coef_ != 0)} / {len(lasso_cv.coef_)}')

In [ ]:
# Side-by-side coefficient comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(feature_names))

axes[0].barh(x_pos, lasso_cv.coef_, color='steelblue', edgecolor='k', linewidth=0.5)
axes[0].set_yticks(x_pos)
axes[0].set_yticklabels(feature_names)
axes[0].set_xlabel('Coefficient value')
axes[0].set_title('LASSO Coefficients')
axes[0].axvline(0, color='black', linewidth=0.8)

axes[1].barh(x_pos, ridge_cv.coef_, color='#e67e22', edgecolor='k', linewidth=0.5)
axes[1].set_yticks(x_pos)
axes[1].set_yticklabels(feature_names)
axes[1].set_xlabel('Coefficient value')
axes[1].set_title('Ridge Coefficients')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('LASSO (Sparse) vs. Ridge (Dense) Coefficients', fontsize=14)
plt.tight_layout()
plt.show()

Notice the visual difference: LASSO has clear zero bars (features it discarded), while Ridge keeps all features active but with small magnitudes. When interpretability matters — knowing *which* features drive the prediction — LASSO is often the better choice.

---
## 13. Cross-Validated Model Comparison

Let's do a fair, apples-to-apples comparison of all our candidate models using 5-fold cross-validation on the training set.

In [ ]:
models = {
    'Linear (degree 1)': make_pipeline(PolynomialFeatures(1, include_bias=False),
                                       LinearRegression()),
    'Poly degree 2': make_pipeline(PolynomialFeatures(2, include_bias=False),
                                   LinearRegression()),
    'Poly degree 3': make_pipeline(PolynomialFeatures(3, include_bias=False),
                                   LinearRegression()),
    'Poly degree 5': make_pipeline(PolynomialFeatures(5, include_bias=False),
                                   LinearRegression()),
    'Poly degree 10 (no reg)': make_pipeline(PolynomialFeatures(10, include_bias=False),
                                              LinearRegression()),
    'LASSO (degree 10)': make_pipeline(PolynomialFeatures(10, include_bias=False),
                                       StandardScaler(),
                                       Lasso(alpha=lasso_cv.alpha_, max_iter=50000)),
    'Ridge (degree 10)': make_pipeline(PolynomialFeatures(10, include_bias=False),
                                       StandardScaler(),
                                       Ridge(alpha=ridge_cv.alpha_)),
}

cv_results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5,
                             scoring='neg_mean_squared_error')
    mse_scores = -scores
    cv_results.append({
        'Model': name,
        'Mean CV MSE': mse_scores.mean(),
        'Std CV MSE': mse_scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values('Mean CV MSE')
print(cv_df.to_string(index=False))

In [ ]:
# Visualize CV results
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(cv_df))

bars = ax.barh(y_pos, cv_df['Mean CV MSE'], xerr=cv_df['Std CV MSE'],
               color='steelblue', edgecolor='k', linewidth=0.5, capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels(cv_df['Model'])
ax.set_xlabel('Mean CV MSE (lower is better)')
ax.set_title('Cross-Validated Model Comparison')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 14. Final Model Evaluation

Based on cross-validation, let's select the best model, retrain on the full training set, and evaluate on the held-out test data one final time.

In [ ]:
# Select the best model from our comparison
best_name = cv_df.iloc[0]['Model']
best_model = models[best_name]
best_model.fit(X_train, y_train)

y_pred_final = best_model.predict(X_test)
final_mse = mean_squared_error(y_test, y_pred_final)
final_r2 = r2_score(y_test, y_pred_final)

print(f'Best model: {best_name}')
print(f'Final Test MSE: {final_mse:.2f}')
print(f'Final Test R²:  {final_r2:.4f}')

In [ ]:
# Final visualization: best model prediction vs. truth
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: fitted curve on data
ax = axes[0]
ax.scatter(X_test, y_test, alpha=0.6, edgecolors='k', linewidths=0.5,
           color='steelblue', s=40, label='Test data')
x_smooth = np.linspace(X.min(), X.max(), 500).reshape(-1, 1)
ax.plot(x_smooth, best_model.predict(x_smooth), color='#e74c3c', linewidth=2,
        label=f'Best model ({best_name})')
ax.plot(x_smooth, true_relationship(x_smooth.ravel()), 'k--', linewidth=1.5,
        alpha=0.5, label='True relationship')
ax.set_xlabel('PM2.5 Concentration (µg/m³)')
ax.set_ylabel('Respiratory Index')
ax.set_title('Best Model Fit')
ax.legend()

# Right panel: predicted vs actual
ax = axes[1]
ax.scatter(y_test, y_pred_final, alpha=0.6, edgecolors='k', linewidths=0.5,
           color='steelblue', s=40)
lims = [min(y_test.min(), y_pred_final.min()) - 2,
        max(y_test.max(), y_pred_final.max()) + 2]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Actual Respiratory Index')
ax.set_ylabel('Predicted Respiratory Index')
ax.set_title(f'Predicted vs. Actual (R² = {final_r2:.3f})')
ax.legend()
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 15. Wrap-Up: Key Takeaways

Here's what we've learned from modeling air quality and respiratory health:

1. **Linear regression is insufficient for non-linear data.** When the true relationship has curvature, a straight line will systematically mispredict.

2. **Polynomial features are a simple way to capture non-linearity.** By adding $x^2, x^3, \ldots$ as features, we stay within the linear regression framework while modeling curves.

3. **More complexity is not always better.** High-degree polynomials can fit the training data almost perfectly but fail on new data. The overfitting curve (train vs. test error) makes this visible.

4. **LASSO regularization performs automatic feature selection.** By penalizing the absolute value of coefficients, LASSO drives unimportant terms to exactly zero. This is especially valuable when you have many candidate features.

5. **Ridge regularization shrinks but keeps all features.** It is useful when you believe all features contribute some signal, but you want to prevent any single coefficient from dominating.

6. **Cross-validation is essential for choosing hyperparameters.** Whether it's the polynomial degree or the regularization strength (alpha), always use CV to select — never rely on training set performance alone.

7. **Scaling matters for regularized models.** Because LASSO and Ridge penalize coefficient magnitudes, features must be on comparable scales (use `StandardScaler`) for the penalty to work fairly.